# Index `handbuch.md` into Qdrant

This notebook reads `indexer/notebooks/handbuch.md`, converts the PDF-page sections into canonical LangChain `Document` objects, and writes them to Qdrant through the existing `IndexingPipeline`.

Important: the indexer treats every run as the authoritative snapshot for the configured collection. After a successful upload it deletes Qdrant points in that collection that were not part of this handbook run. Use a dedicated collection for the handbook, or combine all sources into one run.

## 1. Setup

In [ ]:
import os
import re
import sys
from collections.abc import Iterable
from dataclasses import dataclass
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "indexer"
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Run this notebook from indexer/notebooks or the repository root.")
sys.path.insert(0, str(PROJECT_ROOT))

try:
    from IPython.display import Markdown, display
    from langchain_core.documents import Document
    from truststore import inject_into_ssl
except ModuleNotFoundError as error:
    missing = error.name or str(error)
    raise RuntimeError(
        f"Missing dependency {missing!r}. Start Jupyter with the indexer environment: uv run --directory {PROJECT_ROOT} jupyter lab"
    ) from error

inject_into_ssl()

from src.config.settings import IndexerSettings  # noqa: E402
from src.indexer.pipeline import IndexingPipeline  # noqa: E402

## 2. Configuration

Settings are loaded from `indexer/.env`, environment variables, or `indexer/config.yaml` via the existing `IndexerSettings`. Override `COLLECTION_NAME` below if the handbook should go into a separate Qdrant collection. Pages can be excluded before indexing by number, range, or content pattern.

In [ ]:
HANDBUCH_PATH = PROJECT_ROOT / "notebooks" / "handbuch.md"
COLLECTION_NAME = None  # Example: "fabasoft-handbuch"
RUN_INDEXING = True  # Set to True after checking the collection name below.

# Exclude pages before they are sent to the existing indexer pipeline.
EXCLUDED_PAGES = {}
EXCLUDED_PAGE_RANGES = [(1, 15)]  # Inclusive ranges, for example: [(4, 15)]
EXCLUDED_CONTENT_PATTERNS = [
    r"^#\s*Inhalt\b",
]

if not HANDBUCH_PATH.is_file():
    raise FileNotFoundError(HANDBUCH_PATH)

settings_overrides = {}
if COLLECTION_NAME:
    settings_overrides["collection_name"] = COLLECTION_NAME
settings = IndexerSettings(**settings_overrides)

for key, value in {
    "HTTP_PROXY": settings.http_proxy,
    "HTTPS_PROXY": settings.https_proxy,
    "NO_PROXY": settings.no_proxy,
}.items():
    if value:
        os.environ[key] = value
if not settings.qdrant_url:
    raise ValueError("Qdrant URL is not configured. Set VDB_URL or QDRANT_URL.")
if not settings.openai_api_key:
    raise ValueError("OpenAI API key is not configured. Set OPENAI_API_KEY.")

print(f"Handbook: {HANDBUCH_PATH}")
print(f"Qdrant URL: {settings.qdrant_url}")
print(f"Collection: {settings.collection_name}")
print(f"Indexing mode: {settings.indexing_mode}")
print(f"Dense model: {settings.openai_embedding_model}")
if settings.indexing_mode == "hybrid":
    print(f"Sparse model: {settings.sparse_embedding_model} ({settings.sparse_embedding_language})")

## 3. Markdown Source

The Markdown file contains page markers like `<!-- PDF page 42 -->`. The source emits one canonical document per retained PDF page, so the index metadata keeps the original page number while the existing pipeline still handles chunking, deterministic point IDs, embedding, upsert, and stale-point cleanup.

In [ ]:
PAGE_MARKER = re.compile(r"^<!--\s*PDF page\s+(\d+)\s*-->\s*$", re.MULTILINE)
EXCLUDED_CONTENT_REGEXES = [re.compile(pattern, re.IGNORECASE | re.MULTILINE) for pattern in EXCLUDED_CONTENT_PATTERNS]


def is_excluded_page(page_number: int | None, content: str) -> bool:
    if page_number is not None and page_number in EXCLUDED_PAGES:
        return True
    if page_number is not None and any(start <= page_number <= end for start, end in EXCLUDED_PAGE_RANGES):
        return True
    return any(pattern.search(content) for pattern in EXCLUDED_CONTENT_REGEXES)


@dataclass(frozen=True)
class MarkdownHandbuchSource:
    path: Path

    def load_documents(self) -> Iterable[Document]:
        markdown = self.path.read_text(encoding="utf-8")
        matches = list(PAGE_MARKER.finditer(markdown))
        if not matches:
            content = markdown.strip()
            if content and not is_excluded_page(None, content):
                yield self._document("complete", content, None)
            return

        for index, match in enumerate(matches):
            page_number = int(match.group(1))
            start = match.end()
            end = matches[index + 1].start() if index + 1 < len(matches) else len(markdown)
            content = markdown[start:end].strip()
            if content and not is_excluded_page(page_number, content):
                yield self._document(f"page-{page_number:04d}", content, page_number)

    def _document(self, document_id: str, content: str, page_number: int | None) -> Document:
        metadata = {
            "title": "Fabasoft eGov-Suite 2025 Benutzerhilfe",
            "source_id": "SNOW_EAKTE_HANDBUCH",
            "source": self.path.name,
            "knowledge_base": "eAkte",
            "document_type": "attachment",
        }
        if page_number is not None:
            metadata["page"] = page_number
        return Document(id=f"handbuch-md:{document_id}", page_content=content, metadata=metadata)

## 4. Validate And Preview

In [ ]:
source = MarkdownHandbuchSource(HANDBUCH_PATH)
documents = list(source.load_documents())
if not documents:
    raise RuntimeError("No documents were loaded from handbuch.md.")

document_ids = [document.id for document in documents]
if len(document_ids) != len(set(document_ids)):
    raise RuntimeError("Duplicate source document IDs detected.")

pages = [document.metadata.get("page") for document in documents if "page" in document.metadata]
excluded_pages = sorted(EXCLUDED_PAGES)
for start, end in EXCLUDED_PAGE_RANGES:
    excluded_pages.extend(range(start, end + 1))
excluded_pages = sorted(set(excluded_pages))

print(f"Loaded documents: {len(documents)}")
print(f"Excluded pages by number/range: {excluded_pages or 'none'}")
print(f"Excluded content patterns: {EXCLUDED_CONTENT_PATTERNS or 'none'}")
if pages:
    print(f"Page range: {min(pages)}-{max(pages)}")
print(f"Total characters: {sum(len(document.page_content) for document in documents):,}")
display(Markdown(documents[0].page_content[:2500]))

## 5. Run Indexing

Set `RUN_INDEXING = True` in the configuration cell before running this cell. The pipeline creates embeddings, upserts changed chunks, refreshes unchanged payloads, and prunes stale points from the configured collection.

In [ ]:
if not RUN_INDEXING:
    raise RuntimeError("RUN_INDEXING is False. Set it to True in the configuration cell after verifying the target collection.")

result = IndexingPipeline(settings).run(source)
print("Indexing finished")
print(f"Run ID: {result.run_id}")
print(f"Source documents: {result.source_documents}")
print(f"Chunks upserted: {result.chunks_upserted}")
print(f"Stale points deleted: {result.stale_points_deleted}")